# Callable Zero Valuation

We analyse a 40y-nc-10y Callable Zero compared to a Bullet Bermudan option.

In this notebook, we analyse the valuation of callable zero bonds compared to callable fixed rate bonds.

In [ ]:
using Pkg
Pkg.activate("../.")

In [ ]:
using DataFrames
using DiffFusion
using PlotlyJS
using Printf

We calculate an initial set of ATM volatilities `model_vols_0` used as input for subsequent calculations. 

In [ ]:
yts = DiffFusion.flat_forward("EUR", 0.035)

delta = DiffFusion.flat_parameter([ 1., 10. ])
chi = DiffFusion.flat_parameter([ 0.03, 0.30 ] )
sigma = DiffFusion.backward_flat_volatility(
    "EUR",
    [ 0.0 ],
    0.0080 * ones(2,1),
)

ch = DiffFusion.correlation_holder("Std")
DiffFusion.set_correlation!(ch, "EUR_f_1", "EUR_f_2", 0.55)

m0 = DiffFusion.gaussian_hjm_model(
    "EUR",
    delta,
    chi,
    sigma,
    ch,
    nothing,
    DiffFusion.ZeroRateScaling,
)

option_times = 5.0:5.0:40.0
swap_maturities = vcat(2.0, 5.0:5.0:40.0)
model_vols_0 = DiffFusion.model_implied_volatilties(
    yts,
    m0,
    option_times,
    swap_maturities,
)


## Market Setting

We set up a market with flat yield curve and flat model-implied swaption volatilities as reference volatilities.

### 2-Factor Model with Co-terminal Calibration

This is essentially a cross check. Since we use the model-implied volatilities `model_vols_0` we expect to derive flat input volatilities.

In [ ]:
yts = DiffFusion.flat_forward("EUR", 0.035)

delta = DiffFusion.flat_parameter([ 1., 10. ])
chi = DiffFusion.flat_parameter([ 0.03, 0.30 ] )

ch = DiffFusion.correlation_holder("Std")
DiffFusion.set_correlation!(ch, "EUR_f_1", "EUR_f_2", 0.55)

option_times = 5.0:5.0:40.0
swap_maturities = vcat(2.0, 5.0:5.0:40.0)
# input_vols = 0.0080 * ones(length(option_times), length(swap_maturities))
input_vols = model_vols_0

swap_maturity_indices = [
    [ 2, 8 ],
    [ 2, 7 ],
    [ 2, 6 ],
    [ 2, 5 ],
    [ 2, 4 ],
    [ 2, 3 ],
    [ 1, 2 ],
    [ 1, 2 ],
]

res = DiffFusion.gaussian_hjm_model(
    "EUR",
    delta,
    chi,
    ch,
    option_times,
    swap_maturities,
    input_vols,
    yts,
    max_iter = 10,
    volatility_regularisation = 0.1,
    scaling_type = DiffFusion.ZeroRateScaling,
    swap_maturity_indices = swap_maturity_indices,
)
calib_fit = maximum(abs.([
    res.fit[k, swap_maturity_indices[k][2]]
    for k in eachindex(swap_maturity_indices)
]))
@info "Fit: " * string(calib_fit * 1.0e+4) * "bp"

display(res.fit .* 1.0e+4)
display(res.model.sigma_T.sigma_f.values)

option_times = 1.0:1.0:40.0
swap_maturities = 1.0:1.0:40.0

model_vols = DiffFusion.model_implied_volatilties(
    yts,
    res.model,
    option_times,
    swap_maturities,
)

trace = PlotlyBase.surface(
    x = option_times,
    y = swap_maturities,
    z = model_vols .* 1.0e4,
)
layout = PlotlyBase.Layout(
    scene_xaxis_title = "option_times",
    scene_yaxis_title = "swap_maturities",
    scene_zaxis_title = "implied volatility (bp)",
    width = 1400,
    height = 800,
)
plot(trace, layout)

In [ ]:
option_times = 5.0:5.0:40.0
swap_maturities = vcat(2.0, 5.0:5.0:40.0)

swap_maturity_indices = [
    [ 5, 8 ],
    [ 4, 7 ],
    [ 4, 6 ],
    [ 3, 5 ],
    [ 3, 4 ],
    [ 2, 3 ],
    [ 1, 2 ],
    [ 1, 2 ],
]


idx_list = [
    swap_maturities[swap_maturity_indices[k]]
    for k in eachindex(option_times)
]

hcat(option_times, idx_list)

### 1-Factor Model

We calibrate a 1-factor model to co-terminal swaptions and volatilities from the 2-factor model.

In [ ]:
delta_1f = DiffFusion.flat_parameter([ 1., ])
chi_1f = DiffFusion.flat_parameter([ 0.03, ] )

option_times = 5.0:5.0:40.0
swap_maturities = vcat(2.0, 5.0:5.0:40.0)
# input_vols = 0.0080 * ones(length(option_times), length(swap_maturities))
input_vols = model_vols_0

swap_maturity_indices = [
    [ 8, ],
    [ 7, ],
    [ 6, ],
    [ 5, ],
    [ 4, ],
    [ 3, ],
    [ 2, ],
    [ 1, ],
]

res = DiffFusion.gaussian_hjm_model(
    "EUR",
    delta_1f,
    chi_1f,
    ch,
    option_times,
    swap_maturities,
    input_vols,
    yts,
    max_iter = 10,
    volatility_regularisation = 0.01,
    scaling_type = DiffFusion.DiagonalScaling,
    swap_maturity_indices = swap_maturity_indices
)

display(res.fit)
display(res.model.sigma_T.sigma_f.values)

option_times = 1.0:1.0:40.0
swap_maturities = 1.0:1.0:40.0

model_vols = DiffFusion.model_implied_volatilties(
    yts,
    res.model,
    option_times,
    swap_maturities,
)

trace = PlotlyBase.surface(
    x = option_times,
    y = swap_maturities,
    z = model_vols .* 1.0e+4
)
layout = PlotlyBase.Layout(
    scene_xaxis_title = "option_times",
    scene_yaxis_title = "swap_maturities",
    scene_zaxis_title = "implied volatility (bp)",
    width = 1200,
    height = 800,
)
plot(trace, layout)

## Models, Simulation, Paths

We want to analyse models for different levels of volatilities, mean reversion and correlation. This is implemented by wrapping model setup and simulation into functions `model_path_2f` and `model_path_1f`.

### 2-Factor Model

In [ ]:
ctx = DiffFusion.simple_context("Std", ["EUR"])

function model_path_2f(delta_vol, delta_chi, delta_corr)
    #
    delta = DiffFusion.flat_parameter([ 1., 10. ])
    chi = DiffFusion.flat_parameter([ 0.03, 0.30 ] .+ delta_chi)
    #
    ch = DiffFusion.correlation_holder("Std")
    DiffFusion.set_correlation!(ch, "EUR_f_1", "EUR_f_2", 0.55 + delta_corr)
    #
    option_times = 5.0:5.0:40.0
    swap_maturities = vcat(2.0, 5.0:5.0:40.0)
    #input_vols = (0.0080 + delta_vol) .* ones(length(option_times), length(swap_maturities))
    input_vols = model_vols_0 .+ delta_vol
    
    swap_maturity_indices = [
        [ 2, 8 ],
        [ 2, 7 ],
        [ 2, 6 ],
        [ 2, 5 ],
        [ 2, 4 ],
        [ 2, 3 ],
        [ 1, 2 ],
        [ 1, 2 ],
    ]
    #
    swap_maturity_weights = [
        [ 1, 5 ],
        [ 1, 5 ],
        [ 1, 5 ],
        [ 1, 5 ],
        [ 1, 5 ],
        [ 1, 5 ],
        [ 1, 5 ],
        [ 1, 5 ],
    ]
    #
    res = DiffFusion.gaussian_hjm_model(
        "EUR",
        delta,
        chi,
        ch,
        option_times,
        swap_maturities,
        input_vols,
        yts,
        max_iter = 10,
        volatility_regularisation = 0.1,
        scaling_type = DiffFusion.ZeroRateScaling,
        swap_maturity_indices = swap_maturity_indices,
        swap_maturity_weights = swap_maturity_weights,
    )
    calib_fit = maximum(abs.([
        res.fit[k, swap_maturity_indices[k][2]]
        for k in eachindex(swap_maturity_indices)
    ]))
    @info "Fit: " * string(calib_fit * 1.0e+4) * "bp"
    #
    n_paths = 2^16
    sim = DiffFusion.simple_simulation(
        res.model,
        ch,
        0.0:5.0:40.0,
        n_paths,
        with_progress_bar = false,
    )
    #
    return DiffFusion.path( sim, [yts], ctx,)
end

### 1-Factor Model

In [ ]:
function model_path_1f(delta_vol, delta_chi)
    #
    delta = DiffFusion.flat_parameter([ 1., ])
    chi = DiffFusion.flat_parameter([ 0.03, ] .+ delta_chi )
    #
    ch = DiffFusion.correlation_holder("Std")
    DiffFusion.set_correlation!(ch, "EUR_f_1", "EUR_f_2", 0.55 )
    #
    option_times = 5.0:5.0:40.0
    swap_maturities = vcat(2.0, 5.0:5.0:40.0)
    # input_vols = (0.0080 + delta_vol) .* ones(length(option_times), length(swap_maturities))
    input_vols = model_vols_0 .+ delta_vol
    
    swap_maturity_indices = [
        [ 8, ],
        [ 7, ],
        [ 6, ],
        [ 5, ],
        [ 4, ],
        [ 3, ],
        [ 2, ],
        [ 1, ],
    ]
    #
    res = DiffFusion.gaussian_hjm_model(
        "EUR",
        delta,
        chi,
        ch,
        option_times,
        swap_maturities,
        input_vols,
        yts,
        max_iter = 10,
        volatility_regularisation = 0.01,
        scaling_type = DiffFusion.DiagonalScaling,
        swap_maturity_indices = swap_maturity_indices,
    )
    #
    n_paths = 2^16
    sim = DiffFusion.simple_simulation(
        res.model,
        ch,
        0.0:5.0:40.0,
        n_paths,
        with_progress_bar = false,
    )
    #
    return DiffFusion.path( sim, [yts], ctx,)
end

## Payoffs

We setup functions to generate and value products represented via payoff objects.

### Coupon Bond and Bullet Bermudan 

In [ ]:
function forward_coupon_bond(start_time, end_time, coupon)
    strike = DiffFusion.Pay(DiffFusion.ScalarValue(-1.0), start_time)
    coupons = [
        coupon * DiffFusion.ZeroBond(start_time, T, "EUR")
        for T in start_time+1:1.0:end_time
    ]
    redemption = DiffFusion.ZeroBond(start_time, end_time, "EUR")
    return vcat(strike, coupons, redemption)
end

In [ ]:
function european_npv_cb(p)
    cbos = [
        DiffFusion.Max(sum(forward_coupon_bond(t, 40.0, 0.035)), 0.0)
        for t in 10.0:5.0:35.0
    ]
    cbos_npv = [
        DiffFusion.model_price([payoff], p, nothing, "EUR")
        for payoff in cbos
    ]
    return cbos_npv
end

max_european_npv_cb(p) = maximum(european_npv_cb(p)) 

In [ ]:
function regression_variables(start_time, end_time, strike_rate)
    @assert end_time - start_time >= 2.0
    l_short = DiffFusion.LiborRate(start_time, start_time, start_time + 1.0, "EUR")
    l_long  = DiffFusion.LiborRate(start_time, start_time, end_time, "EUR")
    option  = DiffFusion.Max(l_long - strike_rate, 0.0)
    return [ l_short, l_long, option ]
end

make_regression = (C, O) -> DiffFusion.polynomial_regression(C, O, 2)

In [ ]:
function bermudan_cbo()
    strike_rate = 0.035
    U = sum(forward_coupon_bond(35.0, 40.0, strike_rate))
    V = DiffFusion.Pay(DiffFusion.Max(U, 0.0), 35.0)
    for k in 30.0:-5.0:10.0
        U = DiffFusion.Pay(sum(forward_coupon_bond(k, 40.0, strike_rate)), k)
        Z = regression_variables(k, 40.0, strike_rate)
        V = DiffFusion.AmcMax(k, [V], [U], Z, nothing, make_regression, "EUR")
    end
    return V
end

bermudan_npv(p) = DiffFusion.model_price([bermudan_cbo()], p, nothing, "EUR")

In [ ]:
p_ = model_path_2f(0.0, 0.00, 0.0)

display(european_npv_cb(p_))
display(bermudan_npv(p_))

### Zero Bond and Callable Zero

In [ ]:
function forward_zero_bond(start_time, end_time, coupon)
    redemp_amount = (1.0 + coupon)^end_time
    strike_amount = (1.0 + coupon)^start_time
    #
    redemption = redemp_amount * DiffFusion.ZeroBond(start_time, end_time, "EUR")
    strike = DiffFusion.Pay(DiffFusion.ScalarValue(-strike_amount), start_time)
    return [ strike, redemption]
end

In [ ]:
function european_npv_zb(p)
    zbos = [
        DiffFusion.Max(sum(forward_zero_bond(t, 40.0, 0.035)), 0.0)
        for t in 10.0:5.0:35.0
    ]
    zbos_npv = [
        DiffFusion.model_price([payoff], p, nothing, "EUR")
        for payoff in zbos
    ]
    return zbos_npv
end

max_european_npv_zb(p) = maximum(european_npv_zb(p)) 

In [ ]:
function bermudan_zbo()
    strike_rate = 0.035
    U = sum(forward_zero_bond(35.0, 40.0, strike_rate))
    V = DiffFusion.Pay(DiffFusion.Max(U, 0.0), 35.0)
    for k in 30.0:-5.0:10.0
        U = DiffFusion.Pay(sum(forward_zero_bond(k, 40.0, strike_rate)), k)
        Z = regression_variables(k, 40.0, strike_rate)
        V = DiffFusion.AmcMax(k, [V], [U], Z, nothing, make_regression, "EUR")
    end
    return V
end

bermudan_zero_npv(p) = DiffFusion.model_price([bermudan_zbo()], p, nothing, "EUR")

In [ ]:
p_ = model_path_2f(0.00, 0.00, 0.00)

display(max_european_npv_zb(p_))
display(bermudan_zero_npv(p_))

## Analysis

In this section, we analyse various scenarios for Bullet Bermudan and Callable Zero valuation.

In [ ]:
bar_colors_4 = [ "red", "darkred", "blue", "darkblue"]
bar_colors_2 = [ "darkred", "darkblue"]

function bar_plot(df, title = "", colors = bar_colors_4, yaxis_range = (:auto, :auto), size=(:auto, :auto))
    xaxis_title = names(df)[begin]
    xvalues = df[!,xaxis_title]
    traces = PlotlyBase.AbstractTrace[]
    for (idx, name) in enumerate(names(df)[begin+1:end])
        yvalues = df[!, name]
        trace = bar(
            x = xvalues,
            y = yvalues,
            text = [ @sprintf "%.2f" v for v in yvalues ],
            name = name,
            marker_color = colors[(idx-1) % length(colors) + 1],
        )
        push!(traces, trace)
    end
    layout = PlotlyBase.Layout(
        title = title,
        xaxis_title = xaxis_title,
        yaxis_title = "option price",
        xaxis_tickvals = xvalues,
        yaxis_range = yaxis_range,
        width = size[1],
        height = size[2],
    )
    return plot(traces, layout)
end

### 1-Factor - European and Bermudan

In [ ]:
table = DataFrame()
table[!, "Option"] = [
    "10y - 30y",
    "15y - 25y",
    "20y - 20y",
    "25y - 15y",
    "30y - 10y",
    "35y - 5y",
    "Berm",
]

p_1f_0 = model_path_1f(0.0, 0.0)
p_1f_1 = model_path_1f(0.0010, 0.0)

table[!, "Bullet"] = vcat(
    european_npv_cb(p_1f_0),
    bermudan_npv(p_1f_0)
) .* 100.0
table[!, "Bullet (+10bp vol)"] =  vcat(
    european_npv_cb(p_1f_1),
    bermudan_npv(p_1f_1)
) .* 100.0

table[!, "Zero"] = vcat(
    european_npv_zb(p_1f_0),
    bermudan_zero_npv(p_1f_0)
) .* 100.0
table[!, "Zero (+10bp vol)"] =  vcat(
    european_npv_zb(p_1f_1),
    bermudan_zero_npv(p_1f_1)
) .* 100.0


display(bar_plot(table, "1-Factor Model - Europeans and Bermudan", bar_colors_4, (0, 62), (800, 400)))

### 1-Factor Model - Mean Reversion

In [ ]:
delta_chi = [ -0.02, 0.00, 0.02, 0.05, 0.10 ]
results = Vector{Float64}[]
for d_chi in delta_chi
    p_ = model_path_1f(0.0, d_chi)
    npvs = [
        max_european_npv_cb(p_),
        bermudan_npv(p_),
        max_european_npv_zb(p_),
        bermudan_zero_npv(p_),
    ]
    push!(results, npvs)
end
results = permutedims(hcat(results...)) .* 100
switchv = hcat(results[:,2] - results[:, 1], results[:,4] - results[:, 3])
#
results = hcat(delta_chi, results)
switchv = hcat(delta_chi, switchv)
#
options = DataFrame(results, ["Δχ", "MaxEuropBullet", "BermudanBullet", "MaxEuropZero", "BermudanZero"])
switchv = DataFrame(switchv, ["Δχ", "Bullet", "Zero"])

display(options)
display(switchv)


In [ ]:
display(bar_plot(options, "1-Factor Model - Mean Reversion Sensitivity Options", bar_colors_4, (0, 62), (1200, 450)))
display(bar_plot(switchv, "1-Factor Model - Mean Reversion Sensitivity Switch-Value", bar_colors_2, (0, 40), (1200, 450)))


### 2-Factor Model European and Bermudan

In [ ]:
table = DataFrame()
table[!, "Option"] = [
    "10y - 30y",
    "15y - 25y",
    "20y - 20y",
    "25y - 15y",
    "30y - 10y",
    "35y - 5y",
    "Berm",
]

p_2f_0 = model_path_2f(0.0, 0.0, 0.0)
p_2f_1 = model_path_2f(0.0010, 0.0, 0.0)

table[!, "Bullet"] = vcat(
    european_npv_cb(p_2f_0),
    bermudan_npv(p_2f_0)
) .* 100.0
table[!, "Bullet (+10bp vol)"] =  vcat(
    european_npv_cb(p_2f_1),
    bermudan_npv(p_2f_1)
) .* 100.0

table[!, "Zero"] = vcat(
    european_npv_zb(p_2f_0),
    bermudan_zero_npv(p_2f_0)
) .* 100.0
table[!, "Zero (+10bp vol)"] =  vcat(
    european_npv_zb(p_2f_1),
    bermudan_zero_npv(p_2f_1)
) .* 100.0


display(bar_plot(table, "2-Factor Model - Europeans and Bermudan", bar_colors_4, (0, 62), (800, 400)))

### 2-Factor Model - Mean Reversion

In [ ]:
delta_chi = [ -0.02, 0.0, 0.02, 0.05, 0.10]
results = Vector{Float64}[]
for d_chi in delta_chi
    p_ = model_path_2f(0.0, d_chi, 0.0)
    npvs = [
        max_european_npv_cb(p_),
        bermudan_npv(p_),
        max_european_npv_zb(p_),
        bermudan_zero_npv(p_),
    ]
    push!(results, npvs)
end
results = permutedims(hcat(results...)) .* 100
switchv = hcat(results[:,2] - results[:, 1], results[:,4] - results[:, 3])
#
results = hcat(delta_chi, results)
switchv = hcat(delta_chi, switchv)
#
options = DataFrame(results, ["Δχ", "MaxEuropBullet", "BermudanBullet", "MaxEuropZero", "BermudanZero"])
switchv = DataFrame(switchv, ["Δχ", "Bullet", "Zero"])

display(options)
display(switchv)

In [ ]:
display(bar_plot(options, "2-Factor Model - Mean Reversion Sensitivity Options", bar_colors_4, (0, 62), (1200, 450)))
display(bar_plot(switchv, "2-Factor Model - Mean Reversion Sensitivity Switch-Value", bar_colors_2, (0, 40), (1200, 450)))

### 2-Factor Model - Correlation

In [ ]:
delta_corr = [ -0.55, -0.35, -0.20, 0.0, 0.20, 0.35 ]
results = Vector{Float64}[]
for d_corr in delta_corr
    p_ = model_path_2f(0.0, 0.00, d_corr)
    npvs = [
        max_european_npv_cb(p_),
        bermudan_npv(p_),
        max_european_npv_zb(p_),
        bermudan_zero_npv(p_),
    ]
    push!(results, npvs)
end
results = permutedims(hcat(results...)) .* 100
switchv = hcat(results[:,2] - results[:, 1], results[:,4] - results[:, 3])
#
results = hcat(delta_corr, results)
switchv = hcat(delta_corr, switchv)
#
options = DataFrame(results, ["Δρ", "MaxEuropBullet", "BermudanBullet", "MaxEuropZero", "BermudanZero"])
switchv = DataFrame(switchv, ["Δρ", "Bullet", "Zero"])

display(options)
display(switchv)

In [ ]:
display(bar_plot(options, "2-Factor Model - Correlation Sensitivity Options", bar_colors_4, (0, 62), (1200, 450)))
display(bar_plot(switchv, "2-Factor Model - Correlation Sensitivity Switch-Value", bar_colors_2, (0, 40), (1200, 450)))